# Nova Workshop 2026
<br/>
<img src="https://www.polyestertime.com/wp-content/uploads/2017/01/Nova-Chemical-23-09-2016.jpg" />
<br/><br/>

## Introduction
In this module, we're going to refine our data with vibe coding through a medallion lakehouse structure. This will help us get the data in a state that's easy to work with, manage and fork at various points for added value.

### Goal:
- Build a simple medallion model (Bronze → Silver → Gold) on top of `raw_extruder_events`.
- Create a Gold table `gold_daily_line_kpis` per user.

## Setup
The code in the cell below helps us set the parameters for our refinement pipeline:
- raw: The exact sink for the ingest
- bronze: error controlled and metadata added
- silver: refined and cleaned
- gold: aggregrated

In [0]:
user_schema = "andrij_demo"              # TODO: set to your schema
catalog_name = "nova_workshop"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {user_schema}")

raw_table    = f"{catalog_name}.{user_schema}.raw_extruder_events"
bronze_table = f"{catalog_name}.{user_schema}.bronze_extruder_events"
silver_table = f"{catalog_name}.{user_schema}.silver_extruder_events"
gold_table   = f"{catalog_name}.{user_schema}.gold_daily_line_kpis"

print("Raw    :", raw_table)
print("Bronze :", bronze_table)
print("Silver :", silver_table)
print("Gold   :", gold_table)

## Step 1 – Bronze Table

Bronze = raw data + ingestion metadata.

Adding metadata around the raw data helps Databricks AI (Mosaic) understand and add context to what's going on. Typically, an ingestion timestamp is added at this point to provided additional context and is especially useful if we're doing incremental updates on our data.

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_df = spark.read.table(raw_table).withColumn("ingested_at", current_timestamp())
bronze_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)

## Step 2 – Silver Table (Clean &amp; Standardize)

Use Genie to generate:
- A PySpark transformation from Bronze to Silver.
- Logic to drop obviously bad records (negative pressure, null timestamp).
- Reasonable type casting.

Prompt suggestion (inline Genie on the next cell):

> Transform the Bronze table `${bronze_table}` into a cleaned Silver table `${silver_table}`.
> Drop rows with null timestamps or negative `pressure_bar`.
> Keep the same columns and write back as a managed Delta table.

In [0]:
#TODO: Focus your cursor here and inject the prompt into Genie Code


In [0]:
# PLACEHOLDER: ask Genie to generate this transformation.
#
# Example structure:

# bronze_df = spark.read.table(bronze_table)
# from pyspark.sql.functions import col
# silver_df = (
#     bronze_df
#       .filter(col("timestamp").isNotNull())
#       .filter(col("pressure_bar") >= 0)
# )

# silver_df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

# display(spark.read.table(silver_table).limit(10))

## Step 3 – Gold Table (Daily KPIs per Line)

We aggregate to daily KPIs:

- `avg_motor_temp_c`
- `max_pressure_bar`
- `pct_bad` (percentage of BAD records)
- `alert` when `pct_bad > 5`

Prompt suggestion (inline Genie on the next cell):

> Transform the Silver table `${silver_table}` into a report-ready aggregate Gold table `${gold_table}`.
> The fields to aggregate on should include `avg_motor_temp_c`, `max_pressure_bar`, `pct_bad` as a percentage of bad records and an `alert` field (boolean) if `pct_bad > 3.25`.
> The aggregates need to be grouped by day from the `timestamp` column, and by the `line_id` column

In [0]:
#TODO: Focus your cursor here and inject the prompt into Genie Code


In [0]:
# PLACEHOLDER: ask Genie to generate this transformation.
#
# Example structure:

from pyspark.sql.functions import col, avg, max as spark_max, when, date_format

silver_df = spark.read.table(silver_table)

gold_df = (
    silver_df
    .groupBy(date_format(col("timestamp"), "yyyy-MM-dd").alias("day"), col("line_id"))
    .agg(
        avg(col("motor_temp_c")).alias("avg_motor_temp_c"),
        spark_max(col("pressure_bar")).alias("max_pressure_bar"),
        (100.0 * avg(when(col("quality_flag") == "BAD", 1).otherwise(0))).alias("pct_bad"),
        (avg(when(col("quality_flag") == "BAD", 1).otherwise(0)) > 0.0325).alias("alert")
    )
)

gold_df.write.mode("overwrite").format("delta").saveAsTable(gold_table)

display(spark.read.table(gold_table).limit(10))

## Step 4 – Explore Gold KPIs

Let's have a quick preview of our reporting table. This will be a good starting point to understand how our data is going to present. Gold / aggregate-level data is also referred to as 'downsampled' when we start grouping summary statistcis by various tranches. Typical filtering tranches are recommended to be included in any gold-level tables or views.

In [0]:
from pyspark.sql.functions import col

gold_df = spark.read.table(gold_table)
display(
    gold_df.orderBy(col("day").desc(), col("line_id"))
)